 Minimax, Alpha-Beta Pruning, and Depth-Constrained Search



**Deliverables in this notebook:**
1. The exact specified two-level game tree with terminal utilities `3, 5, 2, 9`.
2. Separate `minimax()` and `alpha_beta()` functions, node scores, branch-pruning logs, and terminal evaluation counts.
3. A maximum search depth of **2** (root at depth 0) and a feature-based security heuristic.
4. A *separate*, deeper example that reaches **non-terminal nodes at depth 2**, explicitly demonstrating the required heuristic evaluation.

All core implementation uses only the Python standard library.


- **Firewall MIN:** `min(3, 5) = 3`.
- **Intrusion Detection MIN:** `min(2, 9) = 2`.
- **MAX decision:** `max(3, 2) = 3`, so select **Firewall**.
- With left-to-right **Alpha-Beta**, the right MIN node sees `2` while the MAX root already has `alpha = 3`, so **terminal utility 9 is pruned**. Minimax visits four terminals; Alpha-Beta visits three.



### 2. Implementation

The program uses:

- `GameNode`: data structure for terminal and non-terminal game-tree nodes.
- `SearchStats`: tracks terminal visits, heuristic calls, node values, and skipped branches.
- `minimax()`: recursively evaluates all children to choose MAX or MIN as appropriate.
- `alpha_beta()`: uses alpha/beta bounds to skip branches that cannot change the decision.
- `heuristic_evaluation()`: estimates a security score when the depth limit is reached **before** a terminal.
- `build_original_tree()` and `build_deeper_tree()`: keep the required base problem separate from the cutoff demonstration.

In [1]:

from dataclasses import dataclass, field
from math import inf
from typing import Dict, List, Optional, Tuple


@dataclass(frozen=True)
class SecurityMetrics:
    """Illustrative measurements of a network's defensive condition."""
    patched_hosts: int
    blocked_attacks: int
    compromised_hosts: int
    exposed_services: int


@dataclass(frozen=True)
class GameNode:
    name: str
    player: Optional[str] = None  # 'MAX', 'MIN', or None for terminal nodes
    children: Tuple['GameNode', ...] = ()
    utility: Optional[int] = None
    metrics: Optional[SecurityMetrics] = None

    def is_terminal(self) -> bool:
        return self.utility is not None


@dataclass
class SearchStats:
    terminals_evaluated: int = 0
    heuristic_evaluations: int = 0
    visited_terminals: List[str] = field(default_factory=list)
    evaluated_heuristics: List[Tuple[str, int]] = field(default_factory=list)
    node_values: Dict[str, int] = field(default_factory=dict)
    pruned_branches: List[str] = field(default_factory=list)


def build_original_tree() -> GameNode:
    """MAX root -> two MIN actions -> four terminals, in given left-right order."""
    return GameNode(
        name='Network Defense', player='MAX', children=(
            GameNode('Firewall (MIN)', player='MIN', children=(
                GameNode('Firewall: attack scenario 1', utility=3),
                GameNode('Firewall: attack scenario 2', utility=5),
            )),
            GameNode('Intrusion Detection (MIN)', player='MIN', children=(
                GameNode('IDS: attack scenario 1', utility=2),
                GameNode('IDS: attack scenario 2', utility=9),
            )),
        ),
    )


def heuristic_evaluation(node: GameNode) -> int:
    """Estimate MAX's security score using observable network indicators.

    score = 2 * patched_hosts + 3 * blocked_attacks
            - 4 * compromised_hosts - exposed_services

    More patched hosts / blocked attacks improve security; compromised hosts
    and exposed services lower it. Weights are illustrative, not learned from
    real cybersecurity data. Called only on nonterminal cutoff states.
    """
    if node.is_terminal():
        return node.utility
    if node.metrics is None:
        raise ValueError('Missing network metrics for nonterminal node: ' + node.name)
    m = node.metrics
    return (2 * m.patched_hosts + 3 * m.blocked_attacks
            - 4 * m.compromised_hosts - m.exposed_services)


def minimax(node: GameNode, stats: SearchStats, depth: int = 0,
            max_depth: Optional[int] = None) -> int:
    """Find the MAX-optimal score, expanding all children left-to-right."""
    if node.is_terminal():
        stats.terminals_evaluated += 1
        stats.visited_terminals.append(node.name)
        return node.utility

    if max_depth is not None and depth >= max_depth:
        value = heuristic_evaluation(node)
        stats.heuristic_evaluations += 1
        stats.evaluated_heuristics.append((node.name, value))
        stats.node_values[node.name] = value
        return value

    if not node.children or node.player not in ('MAX', 'MIN'):
        raise ValueError('Invalid nonterminal game node: ' + node.name)

    child_values = [minimax(child, stats, depth + 1, max_depth)
                    for child in node.children]
    result = max(child_values) if node.player == 'MAX' else min(child_values)
    stats.node_values[node.name] = result
    return result


def alpha_beta(node: GameNode, stats: SearchStats,
               alpha: float = -inf, beta: float = inf,
               depth: int = 0, max_depth: Optional[int] = None) -> int:
    """Alpha-Beta Pruning with left-to-right traversal and explicit prune logs."""
    if node.is_terminal():
        stats.terminals_evaluated += 1
        stats.visited_terminals.append(node.name)
        return node.utility

    if max_depth is not None and depth >= max_depth:
        value = heuristic_evaluation(node)
        stats.heuristic_evaluations += 1
        stats.evaluated_heuristics.append((node.name, value))
        stats.node_values[node.name] = value
        return value

    if not node.children or node.player not in ('MAX', 'MIN'):
        raise ValueError('Invalid nonterminal game node: ' + node.name)

    if node.player == 'MAX':
        best = -inf
        for index, child in enumerate(node.children):
            best = max(best, alpha_beta(child, stats, alpha, beta,
                                        depth + 1, max_depth))
            alpha = max(alpha, best)
            if alpha >= beta:
                for remaining in node.children[index + 1:]:
                    stats.pruned_branches.append(
                        remaining.name + ' (subtree skipped at ' + node.name + ')')
                break
    else:  # MIN
        best = inf
        for index, child in enumerate(node.children):
            best = min(best, alpha_beta(child, stats, alpha, beta,
                                        depth + 1, max_depth))
            beta = min(beta, best)
            if alpha >= beta:
                for remaining in node.children[index + 1:]:
                    stats.pruned_branches.append(
                        remaining.name + ' (subtree skipped at ' + node.name + ')')
                break

    stats.node_values[node.name] = int(best)
    return int(best)


def optimal_action(root: GameNode, stats: SearchStats) -> str:
    """Select the first best root action (all sample root children have values)."""
    return max(root.children, key=lambda child: stats.node_values[child.name]).name


def print_tree(node: GameNode, prefix: str = '', is_last: bool = True,
               at_root: bool = True) -> None:
    """Display each node, preserving left-to-right child order."""
    branch = '' if at_root else ('`-- ' if is_last else '|-- ')
    suffix = ('utility=' + str(node.utility) if node.is_terminal()
              else node.player)
    print(prefix + branch + node.name + ' [' + suffix + ']')
    updated = prefix + ('' if at_root else ('    ' if is_last else '|   '))
    for i, child in enumerate(node.children):
        print_tree(child, updated, i == len(node.children) - 1, False)


def build_deeper_tree() -> GameNode:
    """A separate depth-3 demonstration: all nodes at depth 2 are nonterminal.

    The assignment's exact 3,5,2,9 tree above is not modified. These additional
    illustrative attack-response states make it possible to exercise the
    heuristic at a depth cutoff of 2, which cannot happen in the base tree.
    """
    def attack_response(name: str, metrics: SecurityMetrics,
                        first: int, second: int) -> GameNode:
        return GameNode(name, player='MAX', metrics=metrics, children=(
            GameNode(name + ': outcome 1', utility=first),
            GameNode(name + ': outcome 2', utility=second),
        ))

    return GameNode('Extended Network Defense', player='MAX', children=(
        GameNode('Extended Firewall (MIN)', player='MIN', children=(
            attack_response('Firewall / known attack', SecurityMetrics(3, 3, 1, 1), 4, 6),
            attack_response('Firewall / novel attack', SecurityMetrics(3, 2, 1, 2), 3, 5),
        )),
        GameNode('Extended IDS (MIN)', player='MIN', children=(
            attack_response('IDS / known attack', SecurityMetrics(2, 2, 1, 2), 2, 3),
            attack_response('IDS / novel attack', SecurityMetrics(2, 3, 2, 1), 1, 4),
        )),
    ))


def run_original_example() -> None:
    root = build_original_tree()
    print('=== REQUIRED GAME TREE (utility values 3, 5, 2, 9) ===')
    print_tree(root)

    mm_stats = SearchStats()
    mm_value = minimax(root, mm_stats)
    ab_stats = SearchStats()
    ab_value = alpha_beta(root, ab_stats)

    print('\nMINIMAX')
    print('Firewall MIN value:', mm_stats.node_values['Firewall (MIN)'])
    print('Intrusion Detection MIN value:', mm_stats.node_values['Intrusion Detection (MIN)'])
    print('Optimal MAX action:', optimal_action(root, mm_stats))
    print('Root utility:', mm_value)
    print('Terminals evaluated:', mm_stats.terminals_evaluated)

    print('\nALPHA-BETA PRUNING (left to right)')
    print('Optimal MAX action:', optimal_action(root, ab_stats))
    print('Root utility:', ab_value)
    print('Terminals evaluated:', ab_stats.terminals_evaluated)
    print('Pruned branches:', ab_stats.pruned_branches)
    print('Terminal evaluations saved:',
          mm_stats.terminals_evaluated - ab_stats.terminals_evaluated)

    assert mm_value == ab_value == 3
    assert mm_stats.node_values['Firewall (MIN)'] == 3
    assert mm_stats.node_values['Intrusion Detection (MIN)'] == 2
    assert mm_stats.terminals_evaluated == 4
    assert ab_stats.terminals_evaluated == 3
    assert len(ab_stats.pruned_branches) == 1
    assert 'IDS: attack scenario 2' in ab_stats.pruned_branches[0]

    # Cutoff depth 2 on this ORIGINAL tree reaches real terminal nodes,
    # so the heuristic is intentionally NOT invoked here.
    base_cutoff_stats = SearchStats()
    assert minimax(root, base_cutoff_stats, max_depth=2) == 3
    assert base_cutoff_stats.heuristic_evaluations == 0
    print('\nOriginal tree with cutoff 2: heuristic evaluations = 0'
          ' (all nodes at depth 2 are already terminal).')
    print('Required-tree checks: PASSED')


def run_depth_limited_example() -> None:
    root = build_deeper_tree()
    print('\n=== SEPARATE DEPTH-CONSTRAINT DEMONSTRATION ===')
    print_tree(root)
    print('MAX_DEPTH = 2 (root is at depth 0)')
    print('Heuristic = 2*patched_hosts + 3*blocked_attacks'
          ' - 4*compromised_hosts - exposed_services')

    mm_stats = SearchStats()
    mm_value = minimax(root, mm_stats, max_depth=2)
    ab_stats = SearchStats()
    ab_value = alpha_beta(root, ab_stats, max_depth=2)

    print('\nHeuristic evaluations at depth 2 (Minimax):')
    for name, value in mm_stats.evaluated_heuristics:
        print(' ', name + ':', value)
    print('Example: Firewall / known attack => 2(3) + 3(3) - 4(1) - 1 = 10')

    print('\nDEPTH-LIMITED MINIMAX')
    print('Firewall MIN estimated value:', mm_stats.node_values['Extended Firewall (MIN)'])
    print('IDS MIN estimated value:', mm_stats.node_values['Extended IDS (MIN)'])
    print('Optimal estimated action:', optimal_action(root, mm_stats))
    print('Estimated root score:', mm_value)
    print('Real terminals evaluated:', mm_stats.terminals_evaluated)
    print('Heuristic evaluations:', mm_stats.heuristic_evaluations)

    print('\nDEPTH-LIMITED ALPHA-BETA')
    print('Optimal estimated action:', optimal_action(root, ab_stats))
    print('Estimated root score:', ab_value)
    print('Real terminals evaluated:', ab_stats.terminals_evaluated)
    print('Heuristic evaluations:', ab_stats.heuristic_evaluations)
    print('Pruned branches:', ab_stats.pruned_branches)

    full_stats = SearchStats()
    full_value = minimax(root, full_stats)  # no depth limit, for comparison
    print('\nFULL-DEPTH CHECK ON EXTENDED TREE')
    print('Full-depth root score:', full_value)
    print('Full-depth optimal action:', optimal_action(root, full_stats))
    print('Full-depth terminal evaluations:', full_stats.terminals_evaluated)

    assert (mm_value, ab_value) == (6, 6)
    assert mm_stats.heuristic_evaluations == 4
    assert ab_stats.heuristic_evaluations == 3
    assert mm_stats.terminals_evaluated == ab_stats.terminals_evaluated == 0
    assert len(ab_stats.pruned_branches) == 1
    assert full_value == 5 and full_stats.terminals_evaluated == 8
    print('Depth-limited checks: PASSED')

### 3. Execute the required tree and compare algorithms

The assertions check the expected MAX value, intermediate MIN values, number of visited terminal nodes, and exact skipped branch.

In [2]:
run_original_example()

=== REQUIRED GAME TREE (utility values 3, 5, 2, 9) ===
Network Defense [MAX]
|-- Firewall (MIN) [MIN]
|   |-- Firewall: attack scenario 1 [utility=3]
|   `-- Firewall: attack scenario 2 [utility=5]
`-- Intrusion Detection (MIN) [MIN]
    |-- IDS: attack scenario 1 [utility=2]
    `-- IDS: attack scenario 2 [utility=9]

MINIMAX
Firewall MIN value: 3
Intrusion Detection MIN value: 2
Optimal MAX action: Firewall (MIN)
Root utility: 3
Terminals evaluated: 4

ALPHA-BETA PRUNING (left to right)
Optimal MAX action: Firewall (MIN)
Root utility: 3
Terminals evaluated: 3
Pruned branches: ['IDS: attack scenario 2 (subtree skipped at Intrusion Detection (MIN))']
Terminal evaluations saved: 1

Original tree with cutoff 2: heuristic evaluations = 0 (all nodes at depth 2 are already terminal).
Required-tree checks: PASSED


### 4. Depth constraint and heuristic evaluation

**Maximum search depth:** `2`, counting the MAX root at depth `0`.

The heuristic estimates the defensive utility of a **non-terminal** state using four hypothetical measured indicators:

\[
H(s)=2P+3B-4C-E
\]

where **P** is the number of patched hosts, **B** is the number of blocked attacks, **C** is the number of compromised hosts, and **E** is the number of exposed services. Patching and blocking improve the security score; compromises and exposure reduce it. Coefficients are chosen for this **illustrative example**, not inferred from real security data.

**Why a second tree is necessary:** All four nodes at depth 2 in the *required* tree are already terminal, so searching that tree with a depth limit of 2 will never invoke a non-terminal heuristic. The deeper example below introduces MAX response states at depth 2, and places exact utilities at depth 3. Its leaf values are separately chosen for demonstration, and do not replace the original puzzle's required `3,5,2,9`.

For example, `Firewall / known attack` has `P=3`, `B=3`, `C=1`, `E=1`, resulting in:

\[
H(s)=2(3)+3(3)-4(1)-1=10.
\]

At the cutoff, depth-limited Minimax evaluates all four depth-2 states heuristically, while depth-limited Alpha-Beta evaluates three and prunes the remaining state. The evaluations are *estimates*: no depth-3 terminal is reached. A full-depth run on the deeper tree shows the exact score for comparison.

In [3]:
run_depth_limited_example()


=== SEPARATE DEPTH-CONSTRAINT DEMONSTRATION ===
Extended Network Defense [MAX]
|-- Extended Firewall (MIN) [MIN]
|   |-- Firewall / known attack [MAX]
|   |   |-- Firewall / known attack: outcome 1 [utility=4]
|   |   `-- Firewall / known attack: outcome 2 [utility=6]
|   `-- Firewall / novel attack [MAX]
|       |-- Firewall / novel attack: outcome 1 [utility=3]
|       `-- Firewall / novel attack: outcome 2 [utility=5]
`-- Extended IDS (MIN) [MIN]
    |-- IDS / known attack [MAX]
    |   |-- IDS / known attack: outcome 1 [utility=2]
    |   `-- IDS / known attack: outcome 2 [utility=3]
    `-- IDS / novel attack [MAX]
        |-- IDS / novel attack: outcome 1 [utility=1]
        `-- IDS / novel attack: outcome 2 [utility=4]
MAX_DEPTH = 2 (root is at depth 0)
Heuristic = 2*patched_hosts + 3*blocked_attacks - 4*compromised_hosts - exposed_services

Heuristic evaluations at depth 2 (Minimax):
  Firewall / known attack: 10
  Firewall / novel attack: 6
  IDS / known attack: 4
  IDS / nov

### 5. Brief submission explanation

**Required tree:** The left MIN node returns `3`, and the right MIN node returns `2`. MAX therefore chooses the **left defensive strategy (Firewall)**, obtaining utility **3**. The regular Minimax search evaluates **4** terminal nodes. Alpha-Beta Pruning traverses left-to-right, evaluates **3** terminal nodes, and **prunes the rightmost terminal (9)** because the right MIN node's value is already at most `2`, worse for MAX than the known alternative `3`.

**Depth-limited search:** The limit of `2` stops exploration after the root and two additional layers have been considered. If the state at that limit is *non-terminal*, a heuristic is evaluated rather than descending further. In the extended example, heuristic scores for the four depth-2 states are `10`, `6`, `4`, and `4`, and the estimated MAX result is `6`. The depth-limited Minimax run performs **4 heuristic evaluations** and **0 terminal evaluations**; depth-limited Alpha-Beta performs **3 heuristic evaluations** and **0 terminal evaluations** and prunes the `IDS / novel attack` subtree. Without the depth constraint, the extended tree's exact MAX value is `5` after evaluating **8 terminals**. This demonstrates the search-efficiency trade-off: a cutoff reduces work but replaces exact outcomes with estimates.

The original required tree is not altered by this extended example.